# HW2: Error Analysis with Synthetic Data

In this homework you'll run synthetic queries through your recipe agent, annotate the traces in LangSmith, and use Insights to group failure modes.

**What you'll learn:**
- How to import and reuse your agent from HW1
- How to annotate traces with feedback in the LangSmith UI
- How to run an Insights report to identify failure patterns

## Setup

In [ ]:
import sys
sys.path.insert(0, "../hw1")

from dotenv import load_dotenv

load_dotenv()

## 1. Import the Agent from HW1

In [ ]:
from agent import get_agent

agent = get_agent()

## 2. Load Queries from CSV

The file `queries.csv` contains synthetic queries designed to exercise different failure modes.

> **Note:** If `queries.csv` is empty, populate it with your own test queries before running the next cells.

In [ ]:
import pandas as pd

df = pd.read_csv("queries.csv")
print(f"Loaded {len(df)} queries")
df.head()

## 3. Run Queries Through the Agent

Every invocation is automatically traced to LangSmith.

In [ ]:
from langchain.messages import HumanMessage

results = []

for i, row in df.iterrows():
    query = row["query"]
    print(f"[{i+1}/{len(df)}] {query[:80]}..." if len(query) > 80 else f"[{i+1}/{len(df)}] {query}")

    response = agent.invoke(
        {"messages": [HumanMessage(content=query)]}
    )

    answer = response["messages"][-1].content
    results.append({"query": query, "response": answer})

results_df = pd.DataFrame(results)
print(f"\nFinished running {len(results_df)} queries.")

## 4. Annotate Traces in LangSmith (UI)

Now head to [smith.langchain.com](https://smith.langchain.com) and open your project.

### Steps:
1. **Open the Traces tab** — you should see all the runs from step 3
2. **Click on a trace** to expand it
3. **Add feedback** using the annotation queue or inline feedback buttons:
   - Score the response quality (e.g., 1-5 scale or thumbs up/down)
   - Add a comment describing any issues (wrong recipe, ignores dietary restriction, etc.)
4. **Repeat** for a representative sample (aim for 15-20 annotations)

### Tips:
- Focus on traces where the agent clearly fails or gives suboptimal responses
- Use consistent feedback keys (e.g., `correctness`, `relevance`, `dietary_compliance`)
- You can create an **Annotation Queue** to streamline the process: go to *Datasets & Testing > Annotation Queues > New Queue*

## 5. Run Insights (UI)

After annotating traces, use LangSmith Insights to identify patterns in the failures.

### Steps:
1. Go to your project in LangSmith
2. Click the **Monitor** tab
3. Select a feedback key (e.g., `correctness`)
4. View the distribution of scores across your traces
5. Group by tags or metadata to identify which query types fail most often

### What to look for:
- Are dietary restriction queries consistently scored lower?
- Do certain cuisine types get worse responses?
- Are there patterns in which queries trigger tool errors?